In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix

from warnings import filterwarnings
filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/healthcare-dataset-stroke-data.csv')

In [ ]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [ ]:
df.shape

(5110, 12)

In [ ]:
df['stroke'].value_counts()

,count
stroke,
0,4861
1,249


In [ ]:
df.isnull().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,201


In [ ]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

In [ ]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,28.893237,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1


In [ ]:
le = LabelEncoder()

In [ ]:
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
  df[col] = le.fit_transform(df[col])

In [ ]:
df.tail()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
5105,18234,0,80.0,1,0,1,2,1,83.75,28.893237,2,0
5106,44873,0,81.0,0,0,1,3,1,125.20,40.000000,2,0
5107,19723,0,35.0,0,0,1,3,0,82.99,30.600000,2,0
5108,37544,1,51.0,0,0,1,2,0,166.29,25.600000,1,0
5109,44679,0,44.0,0,0,1,0,1,85.28,26.200000,0,0


In [ ]:
x = df.drop(columns=['id','stroke'], axis=1)
y = df['stroke']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()

In [ ]:
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
smote = SMOTE(random_state=42)
x_res, y_res = smote.fit_resample(x_train, y_train)

In [ ]:
df['stroke'].value_counts()

,count
stroke,
0,4861
1,249


In [ ]:
# Compute scale_pos_weight
scale_pos_weight = (sum(y_train == 0)) / (sum(y_train == 1))

model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # <-- key for imbalance
    eval_metric="logloss",
    random_state=42
)


model.fit(x_res,y_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(x_test)
print("Train Accuracy:", model.score(x_train, y_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Train Accuracy: 0.6298923679060665
Test Accuracy: 0.62426614481409
              precision    recall  f1-score   support

           0       0.99      0.61      0.75       960
           1       0.13      0.89      0.22        62

    accuracy                           0.62      1022
   macro avg       0.56      0.75      0.49      1022
weighted avg       0.94      0.62      0.72      1022

[[583 377]
 [  7  55]]


In [ ]:
import pickle
pickle.dump(model, open('model_sp.pkl', 'wb'))
pickle.dump(scaler, open('scaler_sp.pkl', 'wb'))

In [ ]:
sample_input = [0,	44.0,	0,	0,	1,	0,	1,	85.28,	26.200000,	0	]
sample_input_arr = np.array(sample_input).reshape(1, -1)
sample_input_arr_scaled = scaler.transform(sample_input_arr)
prediction = model.predict(sample_input_arr_scaled)
print(prediction)

if prediction[0] == 0:
    print('Not infect')
else:
    print('disease')


[0]
Not infect
